In [1]:
import sys
import os
import torch
import pandas as pd

# Add the project root directory to sys.path to enable imports from the src folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.wrappers.hf_text import HFTextWrapper
from src.attributors.captum_grad import CaptumGradientsAttributor

# Determine the computation device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Environment Setup Complete.")
print(f"Active Device: {device.upper()}")

Environment Setup Complete.
Active Device: CPU


In [2]:
# --- CONFIGURATION ---
model_id = "distilbert-base-uncased-finetuned-sst-2-english"
# A sentence with mixed sentiments ("ugly" vs "works perfectly") to test model reasoning
input_text = "The interface is ugly but the backend works perfectly."

print(f"--- STARTING ATTRIBUTION TEST ---")

# 1. Initialize the Model Wrapper
# This handles the interaction with the Hugging Face model
print("Initializing model wrapper...")
wrapper = HFTextWrapper(model_id, device)

# 2. Initialize the Attributor
# The attributor wraps the model wrapper to apply the Integrated Gradients algorithm
print("Initializing Captum attributor...")
attributor = CaptumGradientsAttributor(wrapper)

# 3. Perform Attribution
# Calculate feature importance scores for the input text.
# If target_output is None, the method explains the class with the highest probability.
print(f"Calculating gradients for input: '{input_text}'")
output = attributor.attribute(input_text, target_output=None)

print("--- CALCULATION COMPLETED ---")

--- STARTING ATTRIBUTION TEST ---
Initializing model wrapper...
Initializing Captum attributor...
Calculating gradients for input: 'The interface is ugly but the backend works perfectly.'
--- CALCULATION COMPLETED ---


In [3]:
# --- RESULTS ANALYSIS ---

# 1. Interpret the Target Class
# The attribution explain why the model chose this specific class.
target_index = output.target
labels_map = {0: "NEGATIVE", 1: "POSITIVE"}
predicted_label = labels_map.get(target_index, "UNKNOWN")

print(f"Explained Class: {target_index} ({predicted_label})")
print("Interpretation: Positive scores support this class, negative scores oppose it.")

# 2. Construct Data Table
# Combine tokens and their corresponding scores into a DataFrame
data = []
for feature, score in zip(output.input_features, output.heatmap):
    data.append({
        "Token": feature.content,
        "Score": score,
        # Absolute value is useful for sorting by magnitude of impact
        "Magnitude": abs(score) 
    })

df = pd.DataFrame(data)

# 3. Visualization
# Display the dataframe with a color gradient.
# Red/Blue gradients are standard for XAI (Red = Positive contribution, Blue = Negative).
# Note: 'coolwarm' colormap: Red (High +), White (0), Blue (High -)
print("\n--- ATTRIBUTION HEATMAP ---")

# Use pandas styling to highlight important words
styled_df = df.style.background_gradient(cmap='coolwarm', subset=['Score'], vmin=-1, vmax=1)
display(styled_df)

Explained Class: 1 (POSITIVE)
Interpretation: Positive scores support this class, negative scores oppose it.

--- ATTRIBUTION HEATMAP ---


,Token,Score,Magnitude
0,[CLS],0.020768,0.020768
1,the,-0.003273,0.003273
2,interface,-0.070226,0.070226
3,is,-0.014156,0.014156
4,ugly,-0.045038,0.045038
5,but,0.486191,0.486191
6,the,-0.025993,0.025993
7,back,-0.014582,0.014582
8,##end,0.012797,0.012797
9,works,0.423278,0.423278
